# 2 — ONNX export and baseline
Runs in the **Jupyter container on node-serve-model**.

In [ ]:
import os, time
import numpy as np
import torch
import onnxruntime as ort

PT_PATH   = "model_artifacts/smartqueue_ranker.pt"
ONNX_PATH = "model_artifacts/smartqueue_ranker.onnx"

In [ ]:
# Export PyTorch -> ONNX (skipped if create_model.py already did it)
if not os.path.exists(ONNX_PATH):
    model = torch.load(PT_PATH, map_location="cpu", weights_only=False)
    model.eval()
    torch.onnx.export(
        model, torch.randn(1, 64), ONNX_PATH,
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        opset_version=17,
    )
print(ONNX_PATH, "ready")

In [ ]:
def benchmark_session(session, input_dim=64, num_trials=500, batch_size=32):
    name = session.get_inputs()[0].name
    single = np.random.rand(1, input_dim).astype(np.float32)
    for _ in range(20):
        session.run(None, {name: single})
    latencies = []
    for _ in range(num_trials):
        t0 = time.time()
        session.run(None, {name: single})
        latencies.append(time.time() - t0)
    print(f"p50: {np.percentile(latencies, 50)*1000:.2f} ms")
    print(f"p95: {np.percentile(latencies, 95)*1000:.2f} ms")
    print(f"p99: {np.percentile(latencies, 99)*1000:.2f} ms")
    batch = np.random.rand(batch_size, input_dim).astype(np.float32)
    times = []
    for _ in range(200):
        t0 = time.time()
        session.run(None, {name: batch})
        times.append(time.time() - t0)
    print(f"Throughput (batch={batch_size}): {batch_size * 200 / sum(times):.1f} samples/sec")

In [ ]:
session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
print("=== ONNX baseline ===")
benchmark_session(session)